# 02 — Data Wrangling & Preprocessing

Clean Reliance OHLCV data and create leakage-safe time-series features. `Close` is the stock price; a separate `Price` column is not required.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

files = list(Path("../data").glob("*.csv")) or list(Path("data").glob("*.csv"))
if not files:
    raise FileNotFoundError("Place the Reliance OHLCV CSV in the project's data/ folder.")
df = pd.read_csv(files[0])
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for c in ["Open","High","Low","Close","Volume"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Date","Open","High","Low","Close"]).drop_duplicates().sort_values("Date").reset_index(drop=True)
df.head()


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2000-01-03,RELIANCE,EQ,233.05,237.50,251.70,237.50,251.70,251.70,249.37,4456424,1.111319e+14,NaN,NaN,NaN
1,2000-01-04,RELIANCE,EQ,251.70,258.40,271.85,251.30,271.85,271.85,263.52,9487878,2.500222e+14,NaN,NaN,NaN
2,2000-01-05,RELIANCE,EQ,271.85,256.65,287.90,256.65,286.75,282.50,274.79,26833684,7.373697e+14,NaN,NaN,NaN
3,2000-01-06,RELIANCE,EQ,282.50,289.00,300.70,289.00,293.50,294.35,295.45,15682286,4.633254e+14,NaN,NaN,NaN
4,2000-01-07,RELIANCE,EQ,294.35,295.00,317.90,293.00,314.50,314.55,308.91,19870977,6.138388e+14,NaN,NaN,NaN


In [2]:
df["Return_1D"] = df["Close"].pct_change()
df["Return_5D"] = df["Close"].pct_change(5)
df["Return_10D"] = df["Close"].pct_change(10)
df["SMA_5"] = df["Close"].rolling(5).mean()
df["SMA_20"] = df["Close"].rolling(20).mean()
df["Volatility_5D"] = df["Return_1D"].rolling(5).std()
df["Volatility_20D"] = df["Return_1D"].rolling(20).std()
df["Daily_Range"] = (df["High"] - df["Low"]) / df["Close"]
df["Volume_Change"] = df["Volume"].pct_change()
df["Next_Close"] = df["Close"].shift(-1)
df["Direction"] = (df["Next_Close"] > df["Close"]).astype(int)
df = df.dropna().reset_index(drop=True)
print(df.shape)
display(df.head())

(2455, 26)


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,Return_5D,Return_10D,SMA_5,SMA_20,Volatility_5D,Volatility_20D,Daily_Range,Volume_Change,Next_Close,Direction
0,2011-06-01,RELIANCE,EQ,951.85,952.00,958.65,943.65,947.5,946.80,947.83,...,0.045033,0.049842,943.36,935.9075,0.015555,0.013412,0.015843,-0.428913,951.05,1
1,2011-06-02,RELIANCE,EQ,946.80,936.55,954.70,936.55,952.5,951.05,947.09,...,0.018418,0.038831,946.80,935.9625,0.009680,0.013440,0.019084,0.171074,934.60,0
2,2011-06-03,RELIANCE,EQ,951.05,960.50,967.00,931.50,936.0,934.60,951.69,...,-0.011476,0.012348,944.63,934.9400,0.011860,0.013923,0.037984,1.028961,937.75,1
3,2011-06-06,RELIANCE,EQ,934.60,934.65,940.80,928.15,938.6,937.75,935.29,...,-0.001172,0.031174,944.41,933.8625,0.011730,0.013903,0.013490,-0.678193,958.25,1
4,2011-06-07,RELIANCE,EQ,937.75,933.55,960.00,933.55,959.6,958.25,950.55,...,0.006724,0.046410,945.69,934.2375,0.014375,0.014668,0.027602,1.863912,949.20,0


In [3]:
output = Path("../data/reliance_cleaned.csv")
df.to_csv(output, index=False)
print("Saved cleaned dataset:", output)

Saved cleaned dataset: ..\data\reliance_cleaned.csv
